# OPTION A: GLA UNDER THE MATCHED RECIPE (last-day gamble)

**Deadline: Aug 3, 11:59 PM AoE. Abort rule: if Cell 1 does not print GO within ~2 hours of effort, stop and post the Option B reply instead.**

| Cell | What | Time |
|---|---|---|
| 1 | Clean GPU, install PyTorch + flash-linear-attention, smoke test | 10-20 min |
| 2 | Write the GLA training script (matched recipe) | instant |
| 3 | Launch training (background, survives tab close) | instant |
| 4 | Monitor (re-run anytime) | instant |
| 5 | FINAL: stop training, evaluate on the clean held-out set | ~20 min |

Plan: train until ~5 hours before the deadline, then run Cell 5 regardless of progress and
report the actual token count honestly. Send Claude: Cell 1's verdict, then Cell 4 output
after ~1h, then Cell 5's final numbers.

In [6]:
import subprocess, sys, os
def sh(cmd): print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout[-3000:] or "(done)")
os.makedirs("/gla/ckpts", exist_ok=True); print("dirs created")

# attempt 1: plain install, verbose
sh(f"{sys.executable} -m pip install flash-linear-attention 2>&1 | tail -12")
rc = subprocess.run(f"{sys.executable} -c 'import fla; print(\"FLA_OK\")'", shell=True, capture_output=True, text=True)
print(rc.stdout, rc.stderr[-1500:])

if "FLA_OK" not in rc.stdout:
    # attempt 2: GitHub head (tracks new torch/triton fastest)
    sh(f"{sys.executable} -m pip install -U triton 2>&1 | tail -3")
    sh(f"{sys.executable} -m pip install -U git+https://github.com/fla-org/flash-linear-attention 2>&1 | tail -10")
    rc = subprocess.run(f"{sys.executable} -c 'import fla; from fla.models import GLAConfig; print(\"FLA_OK\")'", shell=True, capture_output=True, text=True)
    print(rc.stdout, rc.stderr[-1500:])

print("="*50)
print("VERDICT:", "FLA READY — re-run Cell 1's smoke test, then Cell 2" if "FLA_OK" in rc.stdout else "STILL BLOCKED — paste this whole output to Claude")


dirs created

FLA_OK
 
VERDICT: FLA READY — re-run Cell 1's smoke test, then Cell 2


In [16]:
import os
os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"      # set directly (don't paste the real one in chat)

In [7]:
# ═══ CELL 1: cleanup + install + smoke test (GO / NO-GO) ═══
import subprocess, sys, os
def sh(cmd, show=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr[-2000:] if r.returncode != 0 else "")
    if show: print(out or "(done)")
    return r.returncode, out
sh("pkill -9 -f sanatan_training; sleep 5; nvidia-smi --query-gpu=memory.used --format=csv")
print("Installing (a few minutes)...")
sh("pip install -q -U torch --index-url https://download.pytorch.org/whl/cu126 2>&1 | tail -2")
sh("pip install -q -U triton flash-linear-attention transformers datasets 2>&1 | tail -2")
code = '''
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))
from fla.models import GLAConfig, GLAForCausalLM
cfg = GLAConfig(hidden_size=768, num_hidden_layers=12, num_heads=4, vocab_size=126349, use_cache=False)
m = GLAForCausalLM(cfg).cuda().to(torch.bfloat16)
x = torch.randint(0, 126349, (2, 512), device="cuda")
loss = m(input_ids=x, labels=x).loss
loss.backward()
print("PARAMS_M:", round(sum(p.numel() for p in m.parameters())/1e6, 2))
print("GO")
'''
rc, out = sh(f"{sys.executable} - << 'EOF'\n{code}\nEOF")
print("="*50)
print("VERDICT:", "GO — proceed to Cell 2" if "GO" in out else "NO-GO — paste this entire output to Claude")

(done)
Installing (a few minutes)...


torch 2.12.0+cu130 | cuda: True | NVIDIA RTX PRO 6000 Blackwell Workstation Edition
PARAMS_M: 279.25
GO

VERDICT: GO — proceed to Cell 2


In [8]:
# ═══ CELL 2: write the matched-recipe GLA training script ═══
script = r"""
import os, sys, math, time, json, random, threading, queue
import numpy as np, torch
from fla.models import GLAConfig, GLAForCausalLM
from transformers import AutoTokenizer
from datasets import load_dataset

SL, MICRO, LR, WARMUP = 2048, 32, 6e-4, 500
TOKENS = 3_000_000_000
STEPS = TOKENS // (MICRO * SL)
CKPT = "/gla/ckpts"; os.makedirs(CKPT, exist_ok=True)
torch.backends.cuda.matmul.allow_tf32 = True

tok = AutoTokenizer.from_pretrained("GSAI-ML/LLaDA-8B-Instruct", trust_remote_code=True)
V = len(tok); print(f"Tokenizer vocab={V}", flush=True)

MIX = [("HuggingFaceFW/fineweb", None, "text", 0.52),
       ("bigcode/starcoderdata", "default", "content", 0.28),
       ("uonlp/CulturaX", "zh", "text", 0.11),
       ("HuggingFaceFW/fineweb-edu", "sample-100BT", "text", 0.06),
       ("open-web-math/open-web-math", None, "text", 0.015),
       ("HuggingFaceTB/finemath", "finemath-3plus", "text", 0.015)]
srcs, wts = [], []
for path, sub, key, w in MIX:
    kw = {"split": "train", "streaming": True}
    if sub: kw["name"] = sub
    try:
        srcs.append((iter(load_dataset(path, **kw)), key)); wts.append(w)
        print(f"  data ok: {path}", flush=True)
    except Exception as e:
        print(f"  data SKIP {path}: {e}", flush=True)
wts = [w/sum(wts) for w in wts]

q = queue.Queue(maxsize=8)
def producer():
    rng = random.Random(0); buf = []
    while True:
        i = rng.choices(range(len(srcs)), weights=wts)[0]
        try: text = next(srcs[i][0]).get(srcs[i][1], "")
        except StopIteration: continue
        if not text: continue
        buf.extend(tok(text, add_special_tokens=False)["input_ids"] + [tok.eos_token_id or 0])
        while len(buf) >= MICRO*SL:
            chunk = np.array(buf[:MICRO*SL], dtype=np.int64).reshape(MICRO, SL)
            buf = buf[MICRO*SL:]
            q.put(torch.from_numpy(chunk))
threading.Thread(target=producer, daemon=True).start()

val = torch.from_numpy(np.load("/val_tokens_clean.npy")[:32, :SL].astype(np.int64))

cfg = GLAConfig(hidden_size=768, num_hidden_layers=12, num_heads=4, vocab_size=V, use_cache=False)
model = GLAForCausalLM(cfg).cuda()
n_par = sum(p.numel() for p in model.parameters())
print(f"GLA params: {n_par/1e6:.1f}M  steps: {STEPS}  tok/step: {MICRO*SL}", flush=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.1)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s+1)/WARMUP))

def evaluate():
    model.eval(); losses = []
    with torch.no_grad():
        for i in range(0, len(val), 4):
            b = val[i:i+4].cuda()
            with torch.autocast("cuda", dtype=torch.bfloat16):
                losses.append(model(input_ids=b, labels=b).loss.float().item())
    model.train(); return float(np.mean(losses))

best = 1e9; t0 = time.time(); tok_seen = 0
for step in range(1, STEPS+1):
    b = q.get().cuda()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss = model(input_ids=b, labels=b).loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
    tok_seen += MICRO*SL
    if step % 100 == 0:
        tps = tok_seen/(time.time()-t0)
        print(f"[{step}/{STEPS}] loss={loss.item():.3f} tok={tok_seen/1e9:.2f}B tps={tps/1e3:.1f}k ETA={(STEPS-step)*MICRO*SL/tps/3600:.1f}h", flush=True)
    if step % 500 == 0:
        v = evaluate()
        print(f"  Step {step}: val={v:.4f} tok={tok_seen/1e9:.2f}B", flush=True)
        torch.save({"sd": model.state_dict(), "step": step, "tok": tok_seen, "val": v}, f"{CKPT}/gla_latest.pt")
        if v < best:
            best = v
            torch.save({"sd": model.state_dict(), "step": step, "tok": tok_seen, "val": v}, f"{CKPT}/gla_best.pt")
print("DONE", flush=True)
"""
open("/gla/train_gla.py", "w").write(script)
print("wrote /gla/train_gla.py  (matched: tokenizer, mixture, 65,536 tok/step, AdamW 6e-4/0.9,0.95/wd0.1, warmup 500, 3B budget; in-run val = first 32 seqs of the clean held-out set)")

wrote /gla/train_gla.py  (matched: tokenizer, mixture, 65,536 tok/step, AdamW 6e-4/0.9,0.95/wd0.1, warmup 500, 3B budget; in-run val = first 32 seqs of the clean held-out set)


In [18]:
# ═══ CELL 3: LAUNCH (run ONCE — check Cell 4 before ever re-running) ═══
import subprocess, sys
def sh(cmd): print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout or "(done)")
sh("ps aux | grep train_gla | grep -v grep || echo none_running")
sh(f"cd /gla && HF_HOME=/workspace/.hf_home nohup {sys.executable} -u train_gla.py > /gla/gla.log 2>&1 & echo launched")

root       15890  124  1.3 102500940 10871188 ?  Rl   12:52   2:31 /venv/main/bin/python -u train_gla.py

launched



In [48]:
# ═══ CELL 4: MONITOR (re-run anytime) ═══
import subprocess
def sh(cmd): print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout or "(done)")
sh("tail -12 /gla/gla.log")
sh("grep 'val=' /gla/gla.log | tail -6")
sh("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv")

Tokenizer vocab=126349
  data ok: HuggingFaceFW/fineweb
  data ok: bigcode/starcoderdata
  data ok: uonlp/CulturaX
  data ok: HuggingFaceFW/fineweb-edu
  data ok: open-web-math/open-web-math
  data ok: HuggingFaceTB/finemath
GLA params: 279.3M  steps: 45776  tok/step: 65536
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!

(done)
memory.used [MiB], utilization.gpu [%]
46183 MiB, 100 %



In [ ]:
# ═══ CELL 5: FINAL — stop, evaluate gla_best on the FULL clean set (run ~5h before deadline or at DONE) ═══
import subprocess, json
import numpy as np, torch
def sh(cmd): print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout or "(done)")
sh("pkill -f train_gla; sleep 3; echo stopped")
from fla.models import GLAConfig, GLAForCausalLM
ck = torch.load("/gla/ckpts/gla_best.pt", map_location="cpu")
cfg = GLAConfig(hidden_size=768, num_hidden_layers=12, num_heads=4, vocab_size=126349, use_cache=False)
model = GLAForCausalLM(cfg).cuda(); model.load_state_dict(ck["sd"]); model.eval()
val = torch.from_numpy(np.load("/val_tokens_clean.npy")[:, :2048].astype(np.int64))
losses = []
with torch.no_grad():
    for i in range(0, len(val), 4):
        b = val[i:i+4].cuda()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            losses.append(model(input_ids=b, labels=b).loss.float().item())
final = float(np.mean(losses))
print("="*55)
print(f"GLA (matched recipe): CLEAN-SET LOSS = {final:.4f}")
print(f"  at {ck['tok']/1e9:.2f}B tokens (step {ck['step']}), in-run val at save: {ck['val']:.4f}")
print(f"  reference: p1 finetune 3.951 | p2 10B 3.966 | Linear 3B 4.524")
print("SEND THIS BLOCK TO CLAUDE")

## Decision rules
- **Cell 1 NO-GO after ~2h of trying** -> abandon; post the Option B reply (Claude has the text ready).
- **Training too slow** (ETA in Cell 4 says it will not clear ~1.5B tokens before T-5h) -> still fine: run Cell 5 at T-5h and report the actual token count with the loss-vs-tokens framing.
- **Whatever Cell 5 prints, send it to Claude verbatim** -> final reply drafted around the real number, honest in every branch.